# 📊 Geração de Dados — Simulação de Operação de Cobrança

## 🎯 Objetivo

Simular uma operação de cobrança com 10.000 clientes inadimplentes,
incluindo:

- Estrutura da carteira
- Histórico de acionamentos
- Promessas de pagamento
- Conversão financeira

O objetivo é construir uma base realista para desenvolvimento
de indicadores operacionais e executivos (MIS).



In [7]:
# ==========================================================
# CONFIGURAÇÃO INICIAL DO AMBIENTE
# ==========================================================
# Definição de bibliotecas, controle de reprodutibilidade
# e parâmetros estruturais da operação simulada.
# ==========================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Garantir estrutura de pastas
os.makedirs('../data/raw', exist_ok=True)

# Reprodutibilidade
np.random.seed(42)

# Parâmetros da operação
N_CLIENTES = 10000
N_OPERADORES = 8
PERIODO_DIAS = 90


In [2]:
# IDs
cliente_id = np.arange(1, N_CLIENTES + 1)

# Produtos com estratégia de negócio
produtos = np.random.choice(
    ['Crédito Pessoal', 'Cartão de Crédito', 'Empréstimo Consignado'],
    size=N_CLIENTES,
    p=[0.45, 0.35, 0.20]
)

# Distribuição mais realista de atraso
dias_atraso = np.concatenate([
    np.random.randint(1, 31, int(N_CLIENTES * 0.40)),
    np.random.randint(31, 61, int(N_CLIENTES * 0.30)),
    np.random.randint(61, 91, int(N_CLIENTES * 0.15)),
    np.random.randint(91, 181, int(N_CLIENTES * 0.15))
])

np.random.shuffle(dias_atraso)


In [8]:
# ==========================================================
# ETAPA 1 — MODELAGEM DA CARTEIRA INADIMPLENTE
# ==========================================================

cliente_id = np.arange(1, N_CLIENTES + 1)

# Distribuição estratégica de produtos
produtos = np.random.choice(
    ['Crédito Pessoal', 'Cartão de Crédito', 'Empréstimo Consignado'],
    size=N_CLIENTES,
    p=[0.45, 0.35, 0.20]
)

# Distribuição controlada de atraso
dias_atraso = np.concatenate([
    np.random.randint(1, 31, int(N_CLIENTES * 0.40)),
    np.random.randint(31, 61, int(N_CLIENTES * 0.30)),
    np.random.randint(61, 91, int(N_CLIENTES * 0.15)),
    np.random.randint(91, 181, int(N_CLIENTES * 0.15))
])

np.random.shuffle(dias_atraso)

# Classificação da faixa
def classificar_faixa(dias):
    if dias <= 30:
        return '0-30'
    elif dias <= 60:
        return '31-60'
    elif dias <= 90:
        return '61-90'
    else:
        return '90+'

faixa_atraso = [classificar_faixa(d) for d in dias_atraso]

# Score correlacionado ao atraso
score = []
for d in dias_atraso:
    if d <= 30:
        score.append(np.random.randint(600, 900))
    elif d <= 60:
        score.append(np.random.randint(500, 750))
    elif d <= 90:
        score.append(np.random.randint(400, 650))
    else:
        score.append(np.random.randint(300, 550))

# Valor da dívida por produto
valor_divida = []
for p in produtos:
    if p == 'Cartão de Crédito':
        valor_divida.append(np.random.normal(2500, 800))
    elif p == 'Crédito Pessoal':
        valor_divida.append(np.random.normal(5000, 2000))
    else:
        valor_divida.append(np.random.normal(8000, 3000))

valor_divida = np.abs(np.array(valor_divida)).round(2)

# Operadores
operadores_lista = ['Ana', 'Bruno', 'Carlos', 'Daniela', 'Eduardo', 'Fernanda', 'Gustavo', 'Helena']
operadores = np.random.choice(operadores_lista, size=N_CLIENTES)

# DataFrame final
carteira = pd.DataFrame({
    'cliente_id': cliente_id,
    'produto': produtos,
    'valor_divida': valor_divida,
    'dias_atraso': dias_atraso,
    'faixa_atraso': faixa_atraso,
    'score': score,
    'operador': operadores
})

carteira.to_csv('../data/raw/carteira_clientes.csv', index=False)

carteira.head()


,cliente_id,produto,valor_divida,dias_atraso,faixa_atraso,score,operador
0,1,Crédito Pessoal,3015.38,9,0-30,646,Ana
1,2,Empréstimo Consignado,3376.25,15,0-30,793,Gustavo
2,3,Cartão de Crédito,883.52,12,0-30,823,Eduardo
3,4,Cartão de Crédito,1248.76,51,31-60,642,Ana
4,5,Crédito Pessoal,7323.04,25,0-30,836,Ana


In [9]:
# ==========================================================
# ETAPA 2 — SIMULAÇÃO DE ACIONAMENTOS
# ==========================================================

datas = [datetime.today() - timedelta(days=x) for x in range(PERIODO_DIAS)]

N_ACIONAMENTOS = 35000

acionamentos = pd.DataFrame({
    'cliente_id': np.random.choice(cliente_id, N_ACIONAMENTOS),
    'data_acionamento': np.random.choice(datas, N_ACIONAMENTOS),
    'canal': np.random.choice(['Telefone', 'WhatsApp'], N_ACIONAMENTOS, p=[0.65, 0.35])
})

# Contato baseado no score
acionamentos = acionamentos.merge(carteira[['cliente_id','score']], on='cliente_id', how='left')

def prob_contato(score):
    if score >= 700:
        return np.random.choice([0,1], p=[0.3,0.7])
    elif score >= 500:
        return np.random.choice([0,1], p=[0.5,0.5])
    else:
        return np.random.choice([0,1], p=[0.7,0.3])

acionamentos['contato_efetivo'] = acionamentos['score'].apply(prob_contato)

# Promessa
acionamentos['promessa_pagamento'] = np.where(
    acionamentos['contato_efetivo'] == 1,
    np.random.choice([0,1], size=len(acionamentos), p=[0.6,0.4]),
    0
)

acionamentos.drop(columns=['score'], inplace=True)

acionamentos.to_csv('../data/raw/acionamentos.csv', index=False)

acionamentos.head()


,cliente_id,data_acionamento,canal,contato_efetivo,promessa_pagamento
0,3361,2026-02-14 21:49:12.867668,Telefone,1,0
1,490,2026-02-05 21:49:12.867689,WhatsApp,0,0
2,3761,2026-01-29 21:49:12.868339,WhatsApp,0,0
3,8149,2025-12-29 21:49:12.868397,Telefone,1,0
4,8470,2026-02-21 21:49:12.867640,Telefone,0,0


In [10]:
# ==========================================================
# ETAPA 3 — SIMULAÇÃO DE PAGAMENTOS
# ==========================================================

clientes_promessa = acionamentos[acionamentos['promessa_pagamento'] == 1]['cliente_id'].unique()

N_PAGAMENTOS = int(len(clientes_promessa) * 0.65)

pagamentos = pd.DataFrame({
    'cliente_id': np.random.choice(clientes_promessa, N_PAGAMENTOS),
    'data_pagamento': np.random.choice(datas, N_PAGAMENTOS)
})

pagamentos = pagamentos.merge(carteira[['cliente_id','valor_divida']], on='cliente_id', how='left')

pagamentos['valor_pago'] = pagamentos['valor_divida'] * np.random.uniform(0.4, 1.0, N_PAGAMENTOS)
pagamentos['valor_pago'] = pagamentos['valor_pago'].round(2)

pagamentos = pagamentos[['cliente_id','data_pagamento','valor_pago']]

pagamentos.to_csv('../data/raw/pagamentos.csv', index=False)

pagamentos.head()


,cliente_id,data_pagamento,valor_pago
0,4584,2026-01-03 21:49:12.868391,776.61
1,1655,2025-12-14 21:49:12.868431,3128.96
2,1180,2026-02-12 21:49:12.867673,3157.01
3,2537,2025-12-27 21:49:12.868402,1451.05
4,8108,2026-02-14 21:49:12.867668,1419.82


## ✅ Resultado da Simulação

Foram geradas três bases:

- `carteira_clientes.csv`
- `acionamentos.csv`
- `pagamentos.csv`

Essas bases representam a estrutura operacional de uma área de cobrança
e serão utilizadas para:

- Consolidação dos dados
- Cálculo de KPIs
- Construção de dashboards operacionais e executivos
